# Dealer Matching on Snowflake

**Joreca × Snowflake — hands-on workshop**

This notebook rebuilds steps 1 to 4 of the dealer matching pipeline **inside Snowflake**, on a sample of 1,000 real crawled dealers:

| # | Step | What happens | Table |
|---|------|--------------|-------|
| 1 | **Collect** | Load the crawl output | `DEALERS` |
| 2 | **Locate** | Find each dealer on Google Maps | `DEALER_GOOGLE_MAP` |
| 3 | **Enrich** | Gemini **Batch API** web search: legal name, SIRET, standard address | `DEALER_ADDRESS_STANDARD` |
| 4 | **Resolve** | Keep the Google place only when Gemini confirms it | view `DEALERS_ENRICHED` |
| 5 | **Automate** | Stored procedures + a task graph run steps 2 → 4 end to end | tasks `PIPELINE_*` |

**What replaces what**

| Today | On Snowflake |
|-------|--------------|
| MariaDB tables | Snowflake tables |
| Python scripts on a Linux server + cron | Python UDFs (and Tasks for scheduling) |
| Local files / JSON exports | Internal stage |
| Gemini batch: export → upload → wait → download → re-import, from Linux | Same Batch API, driven by 2 stored procedures (+ a Task) |

### Before you start

- [ ] The admin ran `00-setup/admin_prereqs.sql` once
- [ ] Your role is **`WORKSHOP_DEV`** (user menu, bottom left › *Switch role*)
- [ ] The two files are uploaded to the stage (see Block 0)
- [ ] Your Google Maps and Gemini API keys are at hand

> **Cost:** every Google Maps and Gemini call is billed by Google. `SAMPLE_SIZE` below limits how many dealers are sent to the APIs. Start small.

In [ ]:
SAMPLE_SIZE = 10  # number of dealers sent to the APIs (Google Maps + Gemini)

---
## Block 0 — Setup

### 0.1 Session context
Everything runs with the single workshop role. `WORKSHOP_DB` and `WORKSHOP_WH` (XSMALL, auto-suspend 60 s) were created by the admin.

In [ ]:
%%sql -r setup_ctx
USE ROLE WORKSHOP_DEV;
USE DATABASE WORKSHOP_DB;
USE SCHEMA PUBLIC;
USE WAREHOUSE WORKSHOP_WH;

### 0.2 Stage: file storage inside Snowflake
An internal stage replaces the local folders of the Linux server. Create it, then upload the two files **from your terminal**, at the root of the repo:

```bash
snow stage copy data/dealers_sample.csv @WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE
snow stage copy prompts/gemini_search_agency.txt @WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE/prompts
```

**Expected:** `LIST` shows `dealers_sample.csv` and `prompts/gemini_search_agency.txt`.

In [ ]:
%%sql -r create_stage
CREATE STAGE IF NOT EXISTS WORKSHOP_STAGE;
LIST @WORKSHOP_STAGE;

### 0.3 `DEALERS` — output of step 1 (Collect)
One row per dealer **per marketplace site**. Same name and key as the target architecture.

| Column | Meaning |
|--------|---------|
| `AGENCY_ID`, `SITE` | Key: dealer id on the marketplace + marketplace |
| `AGENCY_NAME`, `ADDRESS`, `CITY`, `ZIP_CODE` | Raw crawl values |
| `SIRET`, `MINISITE_URL` | Often missing in the crawl |
| `GOOGLE_PLACE_ID` | Filled in step 2 |
| `JORECA_ID` | Filled in step 6 (matching, out of scope today) |

> Snowflake does not enforce `PRIMARY KEY` uniqueness (only `NOT NULL`): it documents the model.

In [ ]:
%%sql -r create_dealers
CREATE OR REPLACE TABLE DEALERS (
    AGENCY_ID       VARCHAR,
    SITE            VARCHAR,
    AGENCY_NAME     VARCHAR,
    ADDRESS         VARCHAR,
    CITY            VARCHAR,
    ZIP_CODE        VARCHAR,
    SIRET           VARCHAR,
    MINISITE_URL    VARCHAR,
    GOOGLE_PLACE_ID VARCHAR,   -- step 2
    JORECA_ID       VARCHAR,   -- step 6 (out of scope)
    PRIMARY KEY (AGENCY_ID, SITE)
);

### 0.4 Load the crawl sample
`COPY INTO` is the equivalent of the `INSERT`s into MariaDB. The CSV columns start with `SITE, AGENCY_ID`, so the target columns are listed in that order. `EMPTY_FIELD_AS_NULL` turns empty strings (e.g. missing SIRET) into real `NULL`s.

**Expected:** `rows_loaded = 1000`, `errors_seen = 0`.

In [ ]:
%%sql -r load_dealers
COPY INTO DEALERS (SITE, AGENCY_ID, AGENCY_NAME, ADDRESS, CITY, ZIP_CODE, SIRET, MINISITE_URL)
FROM @WORKSHOP_STAGE/dealers_sample.csv
FILE_FORMAT = (TYPE='CSV' FIELD_OPTIONALLY_ENCLOSED_BY='"' SKIP_HEADER=1 EMPTY_FIELD_AS_NULL=TRUE);

### 0.5 Target tables for steps 2 and 3

| Table | Role | Key |
|-------|------|-----|
| `DEALER_GOOGLE_MAP` | **Rolling cache** of Google Maps answers: a place is fetched once and reused every month | `GOOGLE_PLACE_ID` |
| `DEALER_ADDRESS_STANDARD` | **Per-period workspace** holding the Gemini output for each dealer | `AGENCY_ID`, `SITE` |

In [ ]:
%%sql -r create_targets
CREATE OR REPLACE TABLE DEALER_GOOGLE_MAP (
    GOOGLE_PLACE_ID VARCHAR PRIMARY KEY,
    AI_AGENCY_NAME  VARCHAR,
    GOOGLE_JSON     VARIANT,
    CREATED_AT      TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP()
);

CREATE OR REPLACE TABLE DEALER_ADDRESS_STANDARD (
    AGENCY_ID           VARCHAR,
    SITE                VARCHAR,
    GOOGLE_PLACE_ID     VARCHAR,
    JORECA_ID           VARCHAR,
    IS_REUSED           BOOLEAN DEFAULT FALSE,
    AI_JSON             VARIANT,
    AI_STANDARD_ADDRESS VARCHAR,
    CREATED_AT          TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    PRIMARY KEY (AGENCY_ID, SITE)
);

### 0.6 Check
**Expected:** `DEALERS = 1000`, the two other tables = `0`.

In [ ]:
%%sql -r check_load
SELECT 'DEALERS' AS TBL, COUNT(*) AS N FROM DEALERS
UNION ALL SELECT 'DEALER_GOOGLE_MAP', COUNT(*) FROM DEALER_GOOGLE_MAP
UNION ALL SELECT 'DEALER_ADDRESS_STANDARD', COUNT(*) FROM DEALER_ADDRESS_STANDARD;

---
## Block 1 — Collect: explore `DEALERS`

Before enriching, look at what the crawl gives us: name and address quality, which marketplaces, what is missing.

### 1.1 Preview

In [ ]:
%%sql -r dealers_preview
SELECT * FROM DEALERS LIMIT 10;

### 1.2 Dealers per marketplace

In [ ]:
%%sql -r dealers_by_site
SELECT SITE, COUNT(*) AS N
FROM DEALERS
GROUP BY SITE
ORDER BY N DESC;

### 1.3 SIRET coverage
Dealers without SIRET in the crawl are the ones Gemini has to identify.

In [ ]:
%%sql -r siret_coverage
SELECT COUNT(*)                                AS TOTAL,
       COUNT(SIRET)                            AS WITH_SIRET,
       ROUND(COUNT(SIRET) / COUNT(*) * 100, 1) AS SIRET_PCT
FROM DEALERS;

### 1.4 Typical address issues
Country prefix in the address (`FR-74100`), underscores in city names, missing street or zip code.

In [ ]:
%%sql -r address_issues
SELECT AGENCY_NAME, ADDRESS, CITY, ZIP_CODE
FROM DEALERS
WHERE ADDRESS ILIKE 'FR-%' OR CITY LIKE '%\\_%' OR ZIP_CODE IS NULL OR ADDRESS IS NULL
LIMIT 20;

---
## Block 2 — Locate: Google Maps from Snowflake

Snowflake blocks all outbound network calls by default. Three objects open a **controlled** path to an external API:

| Object | Name | Role |
|--------|------|------|
| Network rule | `GOOGLE_MAPS_RULE` | Which host is reachable: `maps.googleapis.com` only |
| Secret | `GOOGLE_MAPS_API_KEY` | The API key, stored encrypted, read by the code at runtime, never visible in SQL |
| External Access Integration | `GOOGLE_MAPS_EAI` | Binds rule + secret; a UDF must reference it explicitly |

The admin created all three with a placeholder key. You only paste the real key.

> On the Google Cloud project, the **Places API** (`findplacefromtext` + `details`) must be enabled.

### 2.1 Paste the Google Maps key
> ⚠️ **Skip the `ALTER SECRET`** if the key was already set before the workshop: running it unedited replaces the key with the placeholder text.

In [ ]:
%%sql -r eai_google
ALTER SECRET WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_API_KEY
    SET SECRET_STRING = 'PASTE_GOOGLE_MAPS_KEY_HERE';

**Check:** the integration is visible to your role (the key value is never shown).

In [ ]:
%%sql -r check_google_eai
SHOW INTEGRATIONS LIKE 'GOOGLE_MAPS_EAI';

### 2.2 The UDF `FIND_GOOGLE_PLACE`
A Python UDF is your Python script, executed by Snowflake on each row: **no server, no cron, no file I/O**.

- Calls `findplacefromtext` then `details` (language `fr`)
- Returns a `VARIANT` (JSON): we keep the full Google answer and parse it in SQL afterwards
- `status`: `OK` / `NOT_FOUND` (no result) / `ERROR` (bad key, quota, API disabled… see the `error` field)
- On exception, only the error type is returned: `requests` errors embed the URL, which contains the key

In [ ]:
%%sql -r udf_google
CREATE OR REPLACE FUNCTION FIND_GOOGLE_PLACE(AGENCY_NAME VARCHAR, ADDRESS VARCHAR, CITY VARCHAR, ZIP_CODE VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('requests')
HANDLER = 'find_place'
EXTERNAL_ACCESS_INTEGRATIONS = (GOOGLE_MAPS_EAI)
SECRETS = ('gmap_key' = WORKSHOP_DB.PUBLIC.GOOGLE_MAPS_API_KEY)
AS
$$
import _snowflake
import requests

def find_place(agency_name, address, city, zip_code):
    key = _snowflake.get_generic_secret_string('gmap_key')
    query = ', '.join(p for p in [agency_name, address, zip_code, city] if p)
    try:
        r = requests.get(
            "https://maps.googleapis.com/maps/api/place/findplacefromtext/json",
            params={"input": query, "inputtype": "textquery",
                    "fields": "place_id", "language": "fr", "key": key},
            timeout=15)
        r.raise_for_status()
        data = r.json()
        if data.get("status") == "ZERO_RESULTS":
            return {"status": "NOT_FOUND", "query": query}
        if data.get("status") != "OK":
            # REQUEST_DENIED (bad key / API not enabled), OVER_QUERY_LIMIT, INVALID_REQUEST...
            return {"status": "ERROR", "query": query,
                    "error": f"{data.get('status')}: {data.get('error_message', '')}"}
        cands = data["candidates"]
        place_id = cands[0]["place_id"]
        d = requests.get(
            "https://maps.googleapis.com/maps/api/place/details/json",
            params={"place_id": place_id, "language": "fr", "key": key,
                    "fields": "place_id,name,formatted_address,formatted_phone_number,website,geometry,business_status,types"},
            timeout=15)
        d.raise_for_status()
        res = d.json().get("result", {})
        loc = res.get("geometry", {}).get("location", {})
        return {"status": "OK", "query": query, "google_place_id": place_id,
                "name": res.get("name"), "formatted_address": res.get("formatted_address"),
                "phone": res.get("formatted_phone_number"), "website": res.get("website"),
                "latitude": loc.get("lat"), "longitude": loc.get("lng"),
                "business_status": res.get("business_status"), "types": res.get("types", [])}
    except Exception as e:
        # type only: requests errors embed the URL, which contains the API key
        return {"status": "ERROR", "query": query, "error": type(e).__name__}
$$;

### 2.3 Test on 3 rows
Always test on a few rows first: each call is billed.

In [ ]:
%%sql -r test_google
SELECT AGENCY_NAME,
       FIND_GOOGLE_PLACE(AGENCY_NAME, ADDRESS, CITY, ZIP_CODE) AS GMAP
FROM DEALERS
LIMIT 3;

### 2.4 Run on `SAMPLE_SIZE` dealers
The API is called **once per dealer**, the raw answers are kept in `GMAP_RESULTS`, and the next cells reuse this table instead of calling the API again.

`{{SAMPLE_SIZE}}` is the Python variable defined at the top (Jinja templating).

In [ ]:
%%sql -r gmap_results
CREATE OR REPLACE TABLE GMAP_RESULTS AS
SELECT AGENCY_ID, SITE,
       FIND_GOOGLE_PLACE(AGENCY_NAME, ADDRESS, CITY, ZIP_CODE) AS GMAP
FROM (SELECT * FROM DEALERS ORDER BY AGENCY_ID LIMIT {{SAMPLE_SIZE}});

### 2.5 Status check
> If every row is `ERROR`, read the `error` field (usually the key or a disabled API) and fix it before going further.

In [ ]:
%%sql -r gmap_status
SELECT GMAP:status::VARCHAR AS STATUS, COUNT(*) AS N
FROM GMAP_RESULTS
GROUP BY 1;

### 2.6 Store the Place ID on the dealer
`DEALERS.GOOGLE_PLACE_ID`, as in the target architecture.

In [ ]:
%%sql -r fill_place_id
UPDATE DEALERS d
SET GOOGLE_PLACE_ID = g.GMAP:google_place_id::VARCHAR
FROM GMAP_RESULTS g
WHERE d.AGENCY_ID = g.AGENCY_ID AND d.SITE = g.SITE
  AND g.GMAP:status::VARCHAR = 'OK';

### 2.7 Fill the Google Maps cache
- `MERGE … WHEN NOT MATCHED` inserts only the places we do not know yet: re-running next month does not duplicate rows.
- `QUALIFY ROW_NUMBER()` keeps one row per Place ID when several dealers point to the same place (multi-site dealers).

In [ ]:
%%sql -r fill_gmap_cache
MERGE INTO DEALER_GOOGLE_MAP t
USING (
    SELECT GMAP:google_place_id::VARCHAR AS GOOGLE_PLACE_ID,
           GMAP:name::VARCHAR            AS AI_AGENCY_NAME,
           GMAP                          AS GOOGLE_JSON
    FROM GMAP_RESULTS
    WHERE GMAP:status::VARCHAR = 'OK'
    QUALIFY ROW_NUMBER() OVER (PARTITION BY GOOGLE_PLACE_ID ORDER BY AGENCY_ID) = 1
) s
ON t.GOOGLE_PLACE_ID = s.GOOGLE_PLACE_ID
WHEN NOT MATCHED THEN INSERT (GOOGLE_PLACE_ID, AI_AGENCY_NAME, GOOGLE_JSON)
VALUES (s.GOOGLE_PLACE_ID, s.AI_AGENCY_NAME, s.GOOGLE_JSON);

### 2.8 Content of the cache

In [ ]:
%%sql -r gmap_preview
SELECT GOOGLE_PLACE_ID,
       AI_AGENCY_NAME,
       GOOGLE_JSON:formatted_address::VARCHAR AS ADDRESS,
       GOOGLE_JSON:phone::VARCHAR             AS PHONE
FROM DEALER_GOOGLE_MAP;

---
## Block 3 — Enrich: the Gemini **Batch API**, driven from Snowflake

This is the flow Joreca runs in production, moved inside Snowflake:

| Step | Today (Linux server) | On Snowflake |
|------|----------------------|--------------|
| Build the requests | Python script writes a JSONL file | `SUBMIT_GEMINI_BATCH` reads `DEALERS` and builds the JSONL in memory |
| Upload + start the job | Script uploads the file, creates the batch | Same procedure: Files API upload + `batchGenerateContent` |
| Wait | Cron / manual checks | `COLLECT_GEMINI_BATCH` (or a **Task** every 5 min) |
| Download + re-import | Download the result file, parse, insert into MariaDB | Same procedure: download + `MERGE` into `DEALER_ADDRESS_STANDARD` |

- **Same prompt** (`prompts/gemini_search_agency.txt` on the stage), **same Google Search grounding**, one request per dealer.
- **Batch API = 50% of the interactive price.** Asynchronous: target 24 h, a small job usually takes minutes.
- Every job is tracked in `GEMINI_BATCH_JOBS`; the JSONL key → dealer mapping in `GEMINI_BATCH_REQUESTS`.

### 3.1 Paste the Gemini key
> ⚠️ **Skip the `ALTER SECRET`** if the key was already set before the workshop.

In [ ]:
%%sql -r eai_gemini
ALTER SECRET WORKSHOP_DB.PUBLIC.GEMINI_API_KEY
    SET SECRET_STRING = 'PASTE_GEMINI_KEY_HERE';

**Check:** the integration is visible to your role.

In [ ]:
%%sql -r check_gemini_eai
SHOW INTEGRATIONS LIKE 'GEMINI_EAI';

### 3.2 Tracking tables
| Table | Content |
|-------|---------|
| `GEMINI_BATCH_JOBS` | One row per job: Gemini name (`batches/…`), state, counts, submitted / checked / collected timestamps |
| `GEMINI_BATCH_REQUESTS` | The `key` of each JSONL line (`r0`, `r1`…) → `AGENCY_ID`, `SITE` |

In [ ]:
%%sql -r batch_tables
CREATE TABLE IF NOT EXISTS GEMINI_BATCH_JOBS (
    BATCH_NAME      VARCHAR PRIMARY KEY,   -- batches/xxx, returned by Gemini
    DISPLAY_NAME    VARCHAR,
    MODEL           VARCHAR,
    INPUT_FILE      VARCHAR,               -- files/xxx: the uploaded JSONL
    N_REQUESTS      NUMBER,
    STATE           VARCHAR,               -- PENDING / RUNNING / SUCCEEDED / FAILED / CANCELLED / EXPIRED
    BATCH_STATS     VARIANT,               -- request / success / failure counts
    SUBMITTED_AT    TIMESTAMP_NTZ DEFAULT CURRENT_TIMESTAMP(),
    LAST_CHECKED_AT TIMESTAMP_NTZ,
    COLLECTED_AT    TIMESTAMP_NTZ,         -- results loaded into DEALER_ADDRESS_STANDARD
    N_COLLECTED     NUMBER
);

CREATE TABLE IF NOT EXISTS GEMINI_BATCH_REQUESTS (
    BATCH_NAME  VARCHAR,
    REQUEST_KEY VARCHAR,                   -- "key" of the JSONL line (r0, r1, ...)
    AGENCY_ID   VARCHAR,
    SITE        VARCHAR
);

### 3.3 Procedure `SUBMIT_GEMINI_BATCH(n)`
A **stored procedure** (not a UDF): it runs once, reads a table, calls the API and writes tracking rows.

1. Selects `n` dealers **without a Gemini result and not already in a pending job**: re-running is safe.
2. Builds one `GenerateContentRequest` per dealer (system prompt + `Agency Name` / `Address` + `google_search` tool).
3. Uploads the JSONL with the **Files API** (no file on disk: built in memory).
4. Creates the job with `batchGenerateContent` and records it in `GEMINI_BATCH_JOBS`.

Errors (bad key, quota…) are returned as `{"status": "ERROR", "step": …, "error": …}`; nothing is recorded in that case.

In [ ]:
%%sql -r proc_submit
CREATE OR REPLACE PROCEDURE SUBMIT_GEMINI_BATCH(SAMPLE_SIZE NUMBER)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests')
IMPORTS = ('@WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE/prompts/gemini_search_agency.txt')
HANDLER = 'submit'
EXTERNAL_ACCESS_INTEGRATIONS = (GEMINI_EAI)
SECRETS = ('gemini_key' = WORKSHOP_DB.PUBLIC.GEMINI_API_KEY)
AS
$$
import _snowflake
import json
import requests

BASE = "https://generativelanguage.googleapis.com"
import os
import sys

MODEL = "gemini-2.5-flash"

# Joreca's production prompt, versioned as a file on the stage (same file as the interactive UDF)
with open(os.path.join(sys._xoptions["snowflake_import_directory"], "gemini_search_agency.txt"), encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()


def _key():
    return _snowflake.get_generic_secret_string("gemini_key")


def _http_error(r):
    # Clean message only: never echo request headers (they contain the key)
    try:
        msg = r.json().get("error", {}).get("message", "")
    except Exception:
        msg = r.text[:300]
    return f"HTTP {r.status_code}: {msg}"


def _short_state(state):
    # The API returns BATCH_STATE_xxx (REST) or JOB_STATE_xxx (SDK docs): keep the last word
    return (state or "UNKNOWN").split("_")[-1]


def build_jsonl(rows):
    """One line per dealer: {"key": ..., "request": GenerateContentRequest}. Same request as the interactive UDF."""
    lines = []
    for i, row in enumerate(rows):
        lines.append(json.dumps({
            "key": f"r{i}",
            "request": {
                "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
                "contents": [{"role": "user", "parts": [{"text": f"Agency Name: {row['AGENCY_NAME']}\nAddress: {row['ADDR']}"}]}],
                "tools": [{"google_search": {}}],
                "generation_config": {"temperature": 0.1},
            },
        }, ensure_ascii=False))
    return "\n".join(lines) + "\n"


def submit(session, sample_size):
    key = _key()
    # Dealers without a Gemini result yet, and not already waiting in a pending batch
    rows = session.sql(f"""
        SELECT d.AGENCY_ID, d.SITE, d.AGENCY_NAME,
               CONCAT_WS(', ', d.ADDRESS, d.ZIP_CODE, d.CITY, 'France') AS ADDR
        FROM DEALERS d
        WHERE NOT EXISTS (SELECT 1 FROM DEALER_ADDRESS_STANDARD s
                          WHERE s.AGENCY_ID = d.AGENCY_ID AND s.SITE = d.SITE)
          AND NOT EXISTS (SELECT 1 FROM GEMINI_BATCH_REQUESTS r
                          JOIN GEMINI_BATCH_JOBS j ON j.BATCH_NAME = r.BATCH_NAME
                          WHERE r.AGENCY_ID = d.AGENCY_ID AND r.SITE = d.SITE
                            AND j.COLLECTED_AT IS NULL
                            AND j.STATE NOT IN ('FAILED', 'CANCELLED', 'EXPIRED'))
        ORDER BY d.AGENCY_ID
        LIMIT {int(sample_size)}""").collect()
    if not rows:
        return {"status": "NOTHING_TO_DO", "message": "every dealer already has a result or is in a pending batch"}

    data = build_jsonl([r.as_dict() for r in rows]).encode("utf-8")
    display_name = f"joreca-dealers-{len(rows)}"

    # 1. Upload the JSONL file (Files API, resumable protocol: start, then upload + finalize)
    start = requests.post(f"{BASE}/upload/v1beta/files",
                          headers={"x-goog-api-key": key,
                                   "X-Goog-Upload-Protocol": "resumable",
                                   "X-Goog-Upload-Command": "start",
                                   "X-Goog-Upload-Header-Content-Length": str(len(data)),
                                   "X-Goog-Upload-Header-Content-Type": "application/jsonl",
                                   "Content-Type": "application/json"},
                          json={"file": {"display_name": display_name}}, timeout=60)
    upload_url = start.headers.get("x-goog-upload-url")
    if start.status_code != 200 or not upload_url:
        return {"status": "ERROR", "step": "upload start", "error": _http_error(start)}
    up = requests.post(upload_url,
                       headers={"x-goog-api-key": key,
                                "Content-Length": str(len(data)),
                                "X-Goog-Upload-Offset": "0",
                                "X-Goog-Upload-Command": "upload, finalize"},
                       data=data, timeout=300)
    if up.status_code != 200:
        return {"status": "ERROR", "step": "upload", "error": _http_error(up)}
    input_file = up.json()["file"]["name"]

    # 2. Create the batch job on that file
    job = requests.post(f"{BASE}/v1beta/models/{MODEL}:batchGenerateContent",
                        headers={"x-goog-api-key": key},
                        json={"batch": {"display_name": display_name,
                                        "input_config": {"file_name": input_file}}},
                        timeout=60)
    if job.status_code != 200:
        return {"status": "ERROR", "step": "create batch", "error": _http_error(job)}
    op = job.json()
    batch_name = op["name"]
    state = _short_state(op.get("metadata", {}).get("state", "PENDING"))

    # 3. Track the job and the key -> dealer mapping in Snowflake
    session.sql("""INSERT INTO GEMINI_BATCH_JOBS (BATCH_NAME, DISPLAY_NAME, MODEL, INPUT_FILE, N_REQUESTS, STATE)
                   VALUES (?, ?, ?, ?, ?, ?)""",
                params=[batch_name, display_name, MODEL, input_file, len(rows), state]).collect()
    mapping = [[batch_name, f"r{i}", r["AGENCY_ID"], r["SITE"]] for i, r in enumerate(rows)]
    session.create_dataframe(mapping, schema=["BATCH_NAME", "REQUEST_KEY", "AGENCY_ID", "SITE"]) \
           .write.mode("append").save_as_table("GEMINI_BATCH_REQUESTS", column_order="name")

    return {"status": "SUBMITTED", "batch_name": batch_name, "requests": len(rows),
            "input_file": input_file, "state": state}
$$;

### 3.4 Procedure `COLLECT_GEMINI_BATCH(batch_name)`
1. Reads the job state from Gemini (`batches.get`) and updates `GEMINI_BATCH_JOBS`.
2. When the state is **`SUCCEEDED`**: downloads the result file, parses each line (same JSON format as the prompt), maps it back to the dealer with the key.
3. `MERGE` into `DEALER_ADDRESS_STANDARD`: `AI_JSON`, `AI_STANDARD_ADDRESS`, token usage in `AI_JSON:_usage`.

`COLLECT_GEMINI_BATCH(NULL)` processes every job not collected yet: that is what a Task calls.

In [ ]:
%%sql -r proc_collect
CREATE OR REPLACE PROCEDURE COLLECT_GEMINI_BATCH(BATCH_NAME VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('snowflake-snowpark-python', 'requests')
HANDLER = 'collect'
EXTERNAL_ACCESS_INTEGRATIONS = (GEMINI_EAI)
SECRETS = ('gemini_key' = WORKSHOP_DB.PUBLIC.GEMINI_API_KEY)
AS
$$
import _snowflake
import json
import requests

BASE = "https://generativelanguage.googleapis.com"
import re


def _key():
    return _snowflake.get_generic_secret_string("gemini_key")


def _http_error(r):
    # Clean message only: never echo request headers (they contain the key)
    try:
        msg = r.json().get("error", {}).get("message", "")
    except Exception:
        msg = r.text[:300]
    return f"HTTP {r.status_code}: {msg}"


def _short_state(state):
    # The API returns BATCH_STATE_xxx (REST) or JOB_STATE_xxx (SDK docs): keep the last word
    return (state or "UNKNOWN").split("_")[-1]


def _parse_json_text(text):
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text.strip())
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])


def parse_result_line(line):
    """One line of the result file -> (key, result dict). Same result shape as the interactive UDF."""
    obj = json.loads(line)
    key = obj.get("key")
    resp = obj.get("response")
    if resp is None and "candidates" in obj:      # line without wrapper
        resp = obj
    if not resp:
        return key, {"error": json.dumps(obj.get("error") or obj.get("status") or obj)[:500]}
    try:
        text = "".join(p.get("text", "") for p in resp["candidates"][0]["content"]["parts"])
        out = _parse_json_text(text)
    except Exception as e:
        out = {"error": f"{type(e).__name__}: could not parse the model answer"}
    out["_usage"] = resp.get("usageMetadata")
    return key, out


def _collect_one(session, key, batch_name):
    r = requests.get(f"{BASE}/v1beta/{batch_name}", headers={"x-goog-api-key": key}, timeout=60)
    if r.status_code != 200:
        return {"batch_name": batch_name, "status": "ERROR", "error": _http_error(r)}
    op = r.json()
    meta = op.get("metadata", {})
    state = _short_state(meta.get("state"))
    stats = meta.get("batchStats", {})
    session.sql("""UPDATE GEMINI_BATCH_JOBS
                   SET STATE = ?, BATCH_STATS = PARSE_JSON(?), LAST_CHECKED_AT = CURRENT_TIMESTAMP()
                   WHERE BATCH_NAME = ?""",
                params=[state, json.dumps(stats), batch_name]).collect()
    if state != "SUCCEEDED":
        out = {"batch_name": batch_name, "state": state, "stats": stats}
        if op.get("error"):
            out["error"] = op["error"].get("message")
        return out

    # The result file name is in response.responsesFile (REST) or metadata.output.responsesFile
    resp = op.get("response", {})
    result_file = (resp.get("responsesFile")
                   or resp.get("output", {}).get("responsesFile")
                   or meta.get("output", {}).get("responsesFile"))
    if not result_file:
        return {"batch_name": batch_name, "state": state, "status": "ERROR", "error": "no responsesFile in the batch output"}
    dl = requests.get(f"{BASE}/download/v1beta/{result_file}:download",
                      params={"alt": "media"}, headers={"x-goog-api-key": key}, timeout=300)
    if dl.status_code != 200:
        return {"batch_name": batch_name, "state": state, "status": "ERROR", "error": _http_error(dl)}

    keys = {row["REQUEST_KEY"]: (row["AGENCY_ID"], row["SITE"]) for row in session.sql(
        "SELECT REQUEST_KEY, AGENCY_ID, SITE FROM GEMINI_BATCH_REQUESTS WHERE BATCH_NAME = ?",
        params=[batch_name]).collect()}
    results = []
    for line in dl.text.splitlines():
        if not line.strip():
            continue
        k, out = parse_result_line(line)
        if k in keys:
            results.append([keys[k][0], keys[k][1], json.dumps(out, ensure_ascii=False)])

    if results:
        session.create_dataframe(results, schema=["AGENCY_ID", "SITE", "AI_TEXT"]) \
               .write.mode("overwrite").save_as_table("GEMINI_BATCH_LOADED", table_type="temporary")
        # Same target and same columns as the interactive flow: AI_JSON + AI_STANDARD_ADDRESS
        session.sql("""
            MERGE INTO DEALER_ADDRESS_STANDARD t
            USING (SELECT AGENCY_ID, SITE, PARSE_JSON(AI_TEXT) AS AI FROM GEMINI_BATCH_LOADED) s
            ON t.AGENCY_ID = s.AGENCY_ID AND t.SITE = s.SITE
            WHEN MATCHED THEN UPDATE SET
                AI_JSON = s.AI,
                AI_STANDARD_ADDRESS = s.AI:potential_matches[0]:standard_address::VARCHAR,
                IS_REUSED = FALSE
            WHEN NOT MATCHED THEN INSERT (AGENCY_ID, SITE, AI_JSON, AI_STANDARD_ADDRESS, IS_REUSED)
                VALUES (s.AGENCY_ID, s.SITE, s.AI, s.AI:potential_matches[0]:standard_address::VARCHAR, FALSE)
        """).collect()
    session.sql("""UPDATE GEMINI_BATCH_JOBS SET COLLECTED_AT = CURRENT_TIMESTAMP(), N_COLLECTED = ?
                   WHERE BATCH_NAME = ?""", params=[len(results), batch_name]).collect()
    return {"batch_name": batch_name, "state": state, "stats": stats, "collected": len(results)}


def collect(session, batch_name):
    key = _key()
    if batch_name:
        names = [batch_name]
    else:  # all jobs not collected yet and not in a final error state (used by the Task)
        names = [r["BATCH_NAME"] for r in session.sql("""
            SELECT BATCH_NAME FROM GEMINI_BATCH_JOBS
            WHERE COLLECTED_AT IS NULL AND STATE NOT IN ('FAILED', 'CANCELLED', 'EXPIRED')""").collect()]
    try:
        return [_collect_one(session, key, n) for n in names]
    except Exception as e:
        return [{"status": "ERROR", "error": f"{type(e).__name__}: {str(e)[:300]}"}]
$$;

### 3.5 Submit the job
**Expected:** `"status": "SUBMITTED"` and a `batch_name` like `batches/…`.

In [ ]:
%%sql -r submit_batch
CALL SUBMIT_GEMINI_BATCH({{SAMPLE_SIZE}});

### 3.6 ☕ Coffee break — follow the job
States: `PENDING` → `RUNNING` → `SUCCEEDED` (or `FAILED` / `EXPIRED`). Re-run the next two cells after the break.

In [ ]:
%%sql -r collect_batch
CALL COLLECT_GEMINI_BATCH(NULL);

In [ ]:
%%sql -r batch_jobs
SELECT BATCH_NAME, STATE, N_REQUESTS, BATCH_STATS, N_COLLECTED,
       SUBMITTED_AT, LAST_CHECKED_AT, COLLECTED_AT
FROM GEMINI_BATCH_JOBS
ORDER BY SUBMITTED_AT DESC;

**Expected after the break:** `STATE = SUCCEEDED` and `N_COLLECTED` = number of dealers sent.

> **In production**, nobody re-runs this cell: a Task polls every 5 minutes (needs `EXECUTE TASK`, see `admin_prereqs.sql` and optional block 09):
> ```sql
> CREATE OR REPLACE TASK COLLECT_GEMINI_BATCH_TASK
>     WAREHOUSE = WORKSHOP_WH SCHEDULE = '5 MINUTE'
> AS CALL COLLECT_GEMINI_BATCH(NULL);
> ALTER TASK COLLECT_GEMINI_BATCH_TASK RESUME;
> ```

### 3.7 Read the Gemini answers
JSON is parsed directly in SQL with the `:` and `[]` notation.

**Expected:** the `ERROR` column is empty.

In [ ]:
%%sql -r das_preview
SELECT AGENCY_ID, SITE,
       AI_JSON:overall_search_status::VARCHAR                           AS SEARCH_STATUS,
       AI_JSON:potential_matches[0]:found_agency_name::VARCHAR          AS FOUND_NAME,
       AI_STANDARD_ADDRESS,
       AI_JSON:potential_matches[0]:establishment_level_number::VARCHAR AS AI_SIRET,
       AI_JSON:potential_matches[0]:match_confidence_score::FLOAT       AS CONFIDENCE,
       AI_JSON:_usage:promptTokenCount::NUMBER                          AS INPUT_TOKENS,
       AI_JSON:_usage:candidatesTokenCount::NUMBER                      AS OUTPUT_TOKENS,
       AI_JSON:error::VARCHAR                                           AS ERROR
FROM DEALER_ADDRESS_STANDARD;

### 3.8 Token usage per dealer
Real consumption per dealer (batch price = 50% of interactive): the input for the Gemini vs Cortex AI cost comparison (optional block 06).

In [ ]:
%%sql -r token_usage
SELECT COUNT(*)                                                AS CALLS,
       ROUND(AVG(AI_JSON:_usage:promptTokenCount::NUMBER))     AS AVG_INPUT_TOKENS,
       ROUND(AVG(AI_JSON:_usage:candidatesTokenCount::NUMBER)) AS AVG_OUTPUT_TOKENS,
       ROUND(AVG(AI_JSON:_usage:totalTokenCount::NUMBER))      AS AVG_TOTAL_TOKENS
FROM DEALER_ADDRESS_STANDARD
WHERE AI_JSON:error IS NULL;

### 3.9 (Optional) Interactive mode: debug one dealer at a time
While the batch runs, the same prompt can be called **synchronously** with a UDF: useful to iterate on the prompt or debug one dealer (full price, 10–30 s per call). Results are **not** written to `DEALER_ADDRESS_STANDARD`.

In [ ]:
%%sql -r udf_gemini
CREATE OR REPLACE FUNCTION SEARCH_AGENCY(AGENCY_NAME VARCHAR, ADDRESS VARCHAR)
RETURNS VARIANT
LANGUAGE PYTHON
RUNTIME_VERSION = '3.11'
PACKAGES = ('requests')
IMPORTS = ('@WORKSHOP_DB.PUBLIC.WORKSHOP_STAGE/prompts/gemini_search_agency.txt')
HANDLER = 'search'
EXTERNAL_ACCESS_INTEGRATIONS = (GEMINI_EAI)
SECRETS = ('gemini_key' = WORKSHOP_DB.PUBLIC.GEMINI_API_KEY)
AS
$$
import _snowflake
import json
import os
import re
import sys
import requests

MODEL = "gemini-2.5-flash"
# Joreca's production prompt, versioned as a file on the stage
with open(os.path.join(sys._xoptions["snowflake_import_directory"], "gemini_search_agency.txt"), encoding="utf-8") as f:
    SYSTEM_PROMPT = f.read()

def _parse(text):
    text = text.strip()
    text = re.sub(r"^```(?:json)?\s*|\s*```$", "", text)
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

def search(agency_name, address):
    key = _snowflake.get_generic_secret_string('gemini_key')
    user_input = f"Agency Name: {agency_name}\nAddress: {address}"
    body = {
        "system_instruction": {"parts": [{"text": SYSTEM_PROMPT}]},
        "contents": [{"role": "user", "parts": [{"text": user_input}]}],
        "tools": [{"google_search": {}}],          # online research, as in their batch setup
        "generationConfig": {"temperature": 0.1},
    }
    try:
        r = requests.post(
            f"https://generativelanguage.googleapis.com/v1beta/models/{MODEL}:generateContent",
            headers={"x-goog-api-key": key}, json=body, timeout=120)
        if r.status_code != 200:
            return {"error": f"HTTP {r.status_code}: {r.json().get('error', {}).get('message', r.text[:300])}"}
        resp = r.json()
        text = "".join(p.get("text", "") for p in resp["candidates"][0]["content"]["parts"])
        out = _parse(text)
        out["_usage"] = resp.get("usageMetadata")   # input/output tokens, for cost tracking
        return out
    except Exception as e:
        return {"error": f"{type(e).__name__}: {e}"}
$$;

In [ ]:
%%sql -r test_gemini
SELECT AGENCY_NAME,
       SEARCH_AGENCY(AGENCY_NAME, CONCAT_WS(', ', ADDRESS, ZIP_CODE, CITY, 'France')) AS AI
FROM DEALERS
LIMIT 3;

---
## Block 4 — Resolve

Decide which Google Place ID is the right one for each dealer.

**Workshop rule (simplified):** keep the Place ID found in step 2 only if
1. Gemini is confident: `match_confidence_score >= 0.7`, **and**
2. the postal code found by Gemini appears in the Google Maps address.

> Production adds a second Gemini pass for ambiguous cases.

### 4.1 Apply the rule

In [ ]:
%%sql -r resolve
UPDATE DEALER_ADDRESS_STANDARD das
SET GOOGLE_PLACE_ID = d.GOOGLE_PLACE_ID
FROM DEALERS d
JOIN DEALER_GOOGLE_MAP g ON d.GOOGLE_PLACE_ID = g.GOOGLE_PLACE_ID
WHERE das.AGENCY_ID = d.AGENCY_ID AND das.SITE = d.SITE
  AND das.AI_JSON:potential_matches[0]:match_confidence_score::FLOAT >= 0.7
  AND CONTAINS(g.GOOGLE_JSON:formatted_address::VARCHAR,
               das.AI_JSON:potential_matches[0]:cp::VARCHAR);

### 4.2 `DEALERS_ENRICHED`: raw vs Gemini vs Google, side by side
One view joining the 3 tables: `RAW_*` columns come from the crawl, `AI_*` from Gemini, `GOOGLE_*` from Google Maps.

In [ ]:
%%sql -r enriched_view
CREATE OR REPLACE VIEW DEALERS_ENRICHED AS
SELECT d.AGENCY_ID, d.SITE,
       d.AGENCY_NAME                                                        AS RAW_NAME,
       das.AI_JSON:potential_matches[0]:found_agency_name::VARCHAR          AS AI_AGENCY_NAME,
       CONCAT_WS(', ', d.ADDRESS, d.ZIP_CODE, d.CITY)                       AS RAW_ADDRESS,
       das.AI_STANDARD_ADDRESS,
       d.SIRET                                                              AS RAW_SIRET,
       das.AI_JSON:potential_matches[0]:establishment_level_number::VARCHAR AS AI_SIRET,
       das.GOOGLE_PLACE_ID                                                  AS RESOLVED_PLACE_ID,
       g.GOOGLE_JSON:formatted_address::VARCHAR                             AS GOOGLE_ADDRESS,
       g.GOOGLE_JSON:latitude::FLOAT                                        AS LATITUDE,
       g.GOOGLE_JSON:longitude::FLOAT                                       AS LONGITUDE,
       das.AI_JSON:potential_matches[0]:match_confidence_score::FLOAT       AS CONFIDENCE
FROM DEALERS d
JOIN DEALER_ADDRESS_STANDARD das ON d.AGENCY_ID = das.AGENCY_ID AND d.SITE = das.SITE
LEFT JOIN DEALER_GOOGLE_MAP g ON das.GOOGLE_PLACE_ID = g.GOOGLE_PLACE_ID;

### 4.3 Compare

In [ ]:
%%sql -r enriched_preview
SELECT * FROM DEALERS_ENRICHED ORDER BY CONFIDENCE DESC;

### 4.4 Summary
How many dealers were processed, standardized by Gemini, and resolved to a Google place.

In [ ]:
%%sql -r summary
SELECT COUNT(*)                   AS PROCESSED,
       COUNT(AI_STANDARD_ADDRESS) AS STANDARDIZED,
       COUNT(RESOLVED_PLACE_ID)   AS PLACE_ID_RESOLVED
FROM DEALERS_ENRICHED;

---
## Block 5 — Automate it: the whole pipeline, end to end, without a server

So far we ran each step by hand. In production nobody runs cells: each step becomes a **stored procedure**, and **tasks** chain them on a schedule. This replaces the Linux server and its cron.

```
PIPELINE_MONTHLY  (cron: 1st of the month)     CALL RUN_LOCATE(n)            -- step 2, new dealers only
   └─▶ PIPELINE_ENRICH  (AFTER the root)       CALL SUBMIT_GEMINI_BATCH(n)   -- step 3a, sends the batch job

PIPELINE_COLLECT  (every 5 minutes)            CALL RUN_COLLECT_AND_RESOLVE() -- step 3b + step 4 when the job is done
```

**Why two chains?** The Gemini batch is asynchronous: chain 1 *starts* the job, chain 2 *waits* for it and runs the rest.

| Procedure | Wraps | Incremental rule |
|-----------|-------|------------------|
| `RUN_LOCATE(n)` | Block 2 cells | Calls Google only for dealers never located successfully |
| `SUBMIT_GEMINI_BATCH(n)` | Block 3 | Dealers without a result and not in a pending job |
| `RUN_COLLECT_AND_RESOLVE()` | Block 3b + Block 4 | Loads finished jobs, then resolves rows not resolved yet |

> Requires the global `EXECUTE TASK` privilege for `WORKSHOP_DEV` (granted in `admin_prereqs.sql`). Tasks run with the privileges of their owner role.

### 5.1 `RUN_LOCATE(n)` — step 2 as one procedure
Same statements as block 2, in **Snowflake Scripting** (SQL procedure): `INSERT` new Google results, `UPDATE DEALERS`, `MERGE` into the cache.

In [ ]:
%%sql -r proc_run_locate
CREATE OR REPLACE PROCEDURE RUN_LOCATE(MAX_DEALERS NUMBER)
RETURNS VARIANT
LANGUAGE SQL
AS
$$
DECLARE
    n_calls  INTEGER;
    n_placed INTEGER;
    n_cached INTEGER;
BEGIN
    CREATE TABLE IF NOT EXISTS GMAP_RESULTS (AGENCY_ID VARCHAR, SITE VARCHAR, GMAP VARIANT);

    -- 1. Call Google only for dealers never located successfully (new dealers, or previous ERROR)
    INSERT INTO GMAP_RESULTS (AGENCY_ID, SITE, GMAP)
    SELECT AGENCY_ID, SITE, FIND_GOOGLE_PLACE(AGENCY_NAME, ADDRESS, CITY, ZIP_CODE)
    FROM (SELECT d.* FROM DEALERS d
          WHERE NOT EXISTS (SELECT 1 FROM GMAP_RESULTS g
                            WHERE g.AGENCY_ID = d.AGENCY_ID AND g.SITE = d.SITE
                              AND g.GMAP:status::VARCHAR <> 'ERROR')
          ORDER BY d.AGENCY_ID
          LIMIT :MAX_DEALERS);
    n_calls := SQLROWCOUNT;

    -- 2. Store the Place ID on the dealer (same statement as block 2)
    UPDATE DEALERS d
    SET GOOGLE_PLACE_ID = g.GMAP:google_place_id::VARCHAR
    FROM GMAP_RESULTS g
    WHERE d.AGENCY_ID = g.AGENCY_ID AND d.SITE = g.SITE
      AND d.GOOGLE_PLACE_ID IS NULL
      AND g.GMAP:status::VARCHAR = 'OK';
    n_placed := SQLROWCOUNT;

    -- 3. Add new places to the rolling cache (same MERGE as block 2)
    MERGE INTO DEALER_GOOGLE_MAP t
    USING (
        SELECT GMAP:google_place_id::VARCHAR AS GOOGLE_PLACE_ID,
               GMAP:name::VARCHAR            AS AI_AGENCY_NAME,
               GMAP                          AS GOOGLE_JSON
        FROM GMAP_RESULTS
        WHERE GMAP:status::VARCHAR = 'OK'
        QUALIFY ROW_NUMBER() OVER (PARTITION BY GOOGLE_PLACE_ID ORDER BY AGENCY_ID) = 1
    ) s
    ON t.GOOGLE_PLACE_ID = s.GOOGLE_PLACE_ID
    WHEN NOT MATCHED THEN INSERT (GOOGLE_PLACE_ID, AI_AGENCY_NAME, GOOGLE_JSON)
    VALUES (s.GOOGLE_PLACE_ID, s.AI_AGENCY_NAME, s.GOOGLE_JSON);
    n_cached := SQLROWCOUNT;

    RETURN OBJECT_CONSTRUCT('google_calls', n_calls, 'place_ids_set', n_placed, 'new_cached_places', n_cached);
END;
$$;

### 5.2 `RUN_RESOLVE()` — step 4 as one procedure

In [ ]:
%%sql -r proc_run_resolve
CREATE OR REPLACE PROCEDURE RUN_RESOLVE()
RETURNS VARIANT
LANGUAGE SQL
AS
$$
BEGIN
    -- Same rule as block 4, only for rows not resolved yet
    UPDATE DEALER_ADDRESS_STANDARD das
    SET GOOGLE_PLACE_ID = d.GOOGLE_PLACE_ID
    FROM DEALERS d
    JOIN DEALER_GOOGLE_MAP g ON d.GOOGLE_PLACE_ID = g.GOOGLE_PLACE_ID
    WHERE das.AGENCY_ID = d.AGENCY_ID AND das.SITE = d.SITE
      AND das.GOOGLE_PLACE_ID IS NULL
      AND das.AI_JSON:potential_matches[0]:match_confidence_score::FLOAT >= 0.7
      AND CONTAINS(g.GOOGLE_JSON:formatted_address::VARCHAR,
                   das.AI_JSON:potential_matches[0]:cp::VARCHAR);
    RETURN OBJECT_CONSTRUCT('resolved', SQLROWCOUNT);
END;
$$;

### 5.3 `RUN_COLLECT_AND_RESOLVE()` — collect the batch, then resolve
A procedure can call other procedures: this is what the polling task runs.

In [ ]:
%%sql -r proc_run_collect
CREATE OR REPLACE PROCEDURE RUN_COLLECT_AND_RESOLVE()
RETURNS VARIANT
LANGUAGE SQL
AS
$$
DECLARE
    collected VARIANT;
    resolved  VARIANT;
BEGIN
    -- Step 3b: load every finished Gemini batch job, then step 4 on the new rows
    CALL COLLECT_GEMINI_BATCH(NULL) INTO :collected;
    CALL RUN_RESOLVE() INTO :resolved;
    RETURN OBJECT_CONSTRUCT('collect', collected, 'resolve', resolved);
END;
$$;

### 5.4 The task graph
`AFTER` creates the dependency: `PIPELINE_ENRICH` starts only when `PIPELINE_MONTHLY` succeeds. Tasks are created **suspended**.

In [ ]:
%%sql -r create_tasks
-- Chain 1 (monthly): locate new dealers, then submit the Gemini batch job
CREATE OR REPLACE TASK PIPELINE_MONTHLY
    WAREHOUSE = WORKSHOP_WH
    SCHEDULE = 'USING CRON 0 6 1 * * Europe/Paris'   -- 1st of each month, 6:00 Paris time
AS
    CALL RUN_LOCATE({{SAMPLE_SIZE}});

CREATE OR REPLACE TASK PIPELINE_ENRICH
    WAREHOUSE = WORKSHOP_WH
    AFTER PIPELINE_MONTHLY                            -- runs when PIPELINE_MONTHLY succeeds
AS
    CALL SUBMIT_GEMINI_BATCH({{SAMPLE_SIZE}});

-- Chain 2 (polling): the batch is asynchronous, so a second task waits for it
CREATE OR REPLACE TASK PIPELINE_COLLECT
    WAREHOUSE = WORKSHOP_WH
    SCHEDULE = '5 MINUTE'
AS
    CALL RUN_COLLECT_AND_RESOLVE();

### 5.5 Enable the chain
`EXECUTE TASK` runs a suspended root, but **skips suspended child tasks**: resume the child. The root stays suspended, so it only runs when we ask.

In [ ]:
%%sql -r resume_tasks
-- EXECUTE TASK skips suspended child tasks: resume the child, keep the root suspended
-- (the root runs on demand now, and only on its monthly schedule once resumed in production)
ALTER TASK PIPELINE_ENRICH RESUME;
SHOW TASKS LIKE 'PIPELINE_%';

### 5.6 Run the whole pipeline now
One command instead of a cron on Linux. **Expected:** a few seconds later, both `PIPELINE_MONTHLY` and `PIPELINE_ENRICH` are `SUCCEEDED` in the history (also visible in Snowsight › Transformation › Tasks, as a graph).

In [ ]:
%%sql -r execute_pipeline
EXECUTE TASK PIPELINE_MONTHLY;

In [ ]:
%%sql -r task_history
SELECT NAME, STATE, SCHEDULED_TIME, COMPLETED_TIME, RETURN_VALUE, ERROR_MESSAGE
FROM TABLE(INFORMATION_SCHEMA.TASK_HISTORY(
        SCHEDULED_TIME_RANGE_START => DATEADD('hour', -1, CURRENT_TIMESTAMP())))
WHERE NAME LIKE 'PIPELINE_%'
ORDER BY SCHEDULED_TIME DESC;

### 5.7 Collect without waiting for the 5-minute schedule
Runs the polling task once. In production it is resumed (`ALTER TASK PIPELINE_COLLECT RESUME`) and runs every 5 minutes by itself.

In [ ]:
%%sql -r execute_collect
EXECUTE TASK PIPELINE_COLLECT;

### 5.8 Go live / stop
- **Production:** `ALTER TASK PIPELINE_MONTHLY RESUME;` and `ALTER TASK PIPELINE_COLLECT RESUME;`
- **After the workshop:** suspend everything (cell below) so nothing keeps calling the APIs.

In [ ]:
%%sql -r suspend_tasks
ALTER TASK PIPELINE_COLLECT SUSPEND;
ALTER TASK PIPELINE_ENRICH  SUSPEND;
ALTER TASK PIPELINE_MONTHLY SUSPEND;
SHOW TASKS LIKE 'PIPELINE_%';

---
## What's next

| Optional block | Topic |
|----------------|-------|
| `optional-05-matching` | Multi-level matching → `CONCESSION` |
| `optional-06-cortex-ai` | Cortex AI vs Gemini (quality and cost) |
| `optional-07-data-quality` | Data quality checks (DMFs need Enterprise; Standard alternative included) |
| `optional-08-change-tracking` | Track dealer changes month over month |
| `optional-09-scheduling` | More on Tasks: alerts, retries, monitoring |

## Cleanup
After the workshop:
- suspend the tasks (cell 5.8)
- an `ACCOUNTADMIN` runs the **CLEANUP** section at the end of `00-setup/admin_prereqs.sql`: it drops the 3 integrations, `WORKSHOP_DB`, `WORKSHOP_WH` and the `WORKSHOP_DEV` role.